In [1]:
import pandas as pd
import os

supplementary_dir = "results/Supplementary_Files"

os.makedirs(supplementary_dir, exist_ok=True)

In [2]:
def build_supplementary_table(
    mwu_path,
    delong_path,
    variant_column,
    top_column,
    second_column,
    variant_names,
    pair_column,
):
    metric_names = {
        "P_Value": "MWU-pvalue",
        "Adjusted_P_Value": "MWU-adjusted-pvalue",
        "ROC_AUC": "ROC-AUC",
    }

    mwu_df = pd.read_csv(mwu_path, sep="\t")
    delong_df = pd.read_csv(delong_path, sep="\t")

    mwu_metrics = mwu_df.reindex(
        columns=["Dataset", "TF", variant_column, *metric_names]
    )
    ordered_columns = [
        (metric, variant) for variant in variant_names for metric in metric_names
    ]

    variant_values = (
        mwu_metrics[mwu_metrics[variant_column].isin(variant_names)]
        .set_index(["Dataset", "TF", variant_column])[list(metric_names)]
        .unstack(variant_column)
        .reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    )
    variant_values.columns = [
        f"{variant_names[variant]}_{metric_names[metric]}"
        for metric, variant in variant_values.columns
    ]

    delong_columns = [
        "Dataset",
        "TF",
        top_column,
        second_column,
        "DeLong_P_Value",
        "DeLong_P_Value_FDR_BH",
        "DeLong_Significant_FDR_BH",
    ]
    summary = delong_df[delong_columns].copy()
    top_variant = summary[top_column].map(variant_names)
    summary[pair_column] = (
        top_variant
        + ", "
        + summary[second_column].map(variant_names)
    )
    summary["Winner"] = top_variant.where(
        summary["DeLong_Significant_FDR_BH"].eq(True)
    )
    summary = summary.drop(
        columns=[top_column, second_column, "DeLong_Significant_FDR_BH"]
    )

    result = (
        variant_values.reset_index()
        .merge(summary, on=["Dataset", "TF"], how="left")
        .rename(
            columns={
                "DeLong_P_Value": "DeLongs-pvalue",
                "DeLong_P_Value_FDR_BH": "DeLongs-adjusted-pvalue",
            }
        )
    )

    final_columns = (
        ["Dataset", "TF"]
        + list(variant_values.columns)
        + [
            pair_column,
            "DeLongs-pvalue",
            "DeLongs-adjusted-pvalue",
            "Winner",
        ]
    )
    return result[final_columns]

def sort_supplementary_table(table):
    return table.sort_values(["TF", "Dataset"], kind="stable").reset_index(drop=True)

## 1. Methods Supplementary File

In [3]:
method_names = {
    "z-aggregate": "z-aggregate",
    "viper": "viper",
    "ulm": "ulm",
    "zscore": "zscore",
}

methods_delongs_df = build_supplementary_table(
    mwu_path="results/Methods_MWU-Delongs/MWU_merged.tsv",
    delong_path="results/Methods_MWU-Delongs/DeLong_top2_merged.tsv",
    variant_column="Method",
    top_column="Top_Method",
    second_column="Second_Method",
    variant_names=method_names,
    pair_column="Top, Second",
)

methods_delongs_df = sort_supplementary_table(methods_delongs_df)

methods_delongs_df.to_csv(
    f"{supplementary_dir}/Methods_Delongs_Comparison_Table.tsv", sep="\t", index=False
)
methods_delongs_df

,Dataset,TF,z-aggregate_MWU-pvalue,z-aggregate_MWU-adjusted-pvalue,z-aggregate_ROC-AUC,viper_MWU-pvalue,viper_MWU-adjusted-pvalue,viper_ROC-AUC,ulm_MWU-pvalue,ulm_MWU-adjusted-pvalue,ulm_ROC-AUC,zscore_MWU-pvalue,zscore_MWU-adjusted-pvalue,zscore_ROC-AUC,"Top, Second",DeLongs-pvalue,DeLongs-adjusted-pvalue,Winner
0,NadigOConner2024_hepg2,ACTL6A,0.183499,0.330298,0.586905,0.293275,0.507591,0.552396,0.199073,0.373262,0.581398,0.199073,0.365644,0.581398,NaN,NaN,NaN,NaN
1,NadigOConner2024_jurkat,ACTL6A,0.000005,0.000079,0.926824,0.000129,0.001828,0.851774,0.000017,0.000297,0.898433,0.000017,0.000297,0.898433,"z-aggregate, ulm",0.054064,0.391965,NaN
2,ReplogleWeissman2022_K562_essential,ACTL6A,0.368066,0.659764,0.513397,0.497003,0.765385,0.500300,0.384532,0.813576,0.511672,0.384532,0.811377,0.511672,NaN,NaN,NaN,NaN
3,ReplogleWeissman2022_K562_gwps,ACTL6A,0.150781,0.493864,0.538842,0.200727,0.553224,0.531545,0.242642,0.712169,0.526237,0.242642,0.685463,0.526237,NaN,NaN,NaN,NaN
4,ReplogleWeissman2022_rpe1,ACTL6A,0.091822,0.148584,0.599176,0.168100,0.257947,0.571734,0.134509,0.213773,0.582447,0.134509,0.217660,0.582447,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,NadigOConner2024_jurkat,ZBTB17,0.017037,0.062964,0.530956,0.492099,0.677017,0.500290,0.576016,0.789700,0.497194,0.964195,1.000000,0.473626,"z-aggregate, viper",0.177829,0.483589,NaN
665,ReplogleWeissman2022_K562_essential,ZBTB17,0.987185,0.999916,0.448527,0.001624,0.007354,0.567887,0.003406,0.021853,0.562412,0.026748,0.121151,0.544536,"viper, ulm",0.380203,0.586267,NaN
666,ReplogleWeissman2022_K562_gwps,ZBTB17,0.989105,0.999999,0.462619,0.142793,0.468128,0.517405,0.067709,0.347779,0.524336,0.307051,0.709274,0.508219,NaN,NaN,NaN,NaN
667,ReplogleWeissman2022_rpe1,ZBTB17,0.681783,0.749119,0.491731,0.424039,0.563276,0.503351,0.320604,0.438980,0.508152,0.628440,0.767606,0.494267,NaN,NaN,NaN,NaN


## 2. Weights Supplementary File

In [4]:
weight_names = {
    "CORRELATION": "Correlation",
    "NONZERORATE": "Nonzero-Rate",
    "SPECIFICITY": "Specificity",
    "UNIFORM": "Uniform",
}

weights_delongs_df = build_supplementary_table(
    mwu_path="results/Weights_MWU-Delongs/MWU_merged.tsv",
    delong_path="results/Weights_MWU-Delongs/DeLong_top2_merged.tsv",
    variant_column="Weight",
    top_column="Top_Weight",
    second_column="Second_Weight",
    variant_names=weight_names,
    pair_column="Top, Second",
)

weights_delongs_df = sort_supplementary_table(weights_delongs_df)

weights_delongs_df.to_csv(
    f"{supplementary_dir}/Weights_Delongs_Comparison_Table.tsv", sep="\t", index=False
)
weights_delongs_df

,Dataset,TF,Correlation_MWU-pvalue,Correlation_MWU-adjusted-pvalue,Correlation_ROC-AUC,Nonzero-Rate_MWU-pvalue,Nonzero-Rate_MWU-adjusted-pvalue,Nonzero-Rate_ROC-AUC,Specificity_MWU-pvalue,Specificity_MWU-adjusted-pvalue,Specificity_ROC-AUC,Uniform_MWU-pvalue,Uniform_MWU-adjusted-pvalue,Uniform_ROC-AUC,"Top, Second",DeLongs-pvalue,DeLongs-adjusted-pvalue,Winner
0,NadigOConner2024_hepg2,ACTL6A,0.089546,0.187421,0.629427,0.116961,0.250631,0.614666,0.162298,0.304309,0.594895,0.183499,0.330298,0.586905,NaN,NaN,NaN,NaN
1,NadigOConner2024_jurkat,ACTL6A,0.000007,0.000142,0.919190,0.000009,0.000126,0.912582,0.000008,0.000096,0.915676,0.000005,0.000079,0.926824,"Uniform, Correlation",0.095103,0.244162,NaN
2,ReplogleWeissman2022_K562_essential,ACTL6A,0.400088,0.733495,0.510063,0.464466,0.674791,0.503546,0.444325,0.698225,0.505567,0.368066,0.659764,0.513397,NaN,NaN,NaN,NaN
3,ReplogleWeissman2022_K562_gwps,ACTL6A,0.052320,0.295608,0.561012,0.044371,0.204651,0.563993,0.162763,0.540948,0.536965,0.150781,0.493864,0.538842,NaN,NaN,NaN,NaN
4,ReplogleWeissman2022_rpe1,ACTL6A,0.086222,0.156608,0.601770,0.109980,0.183230,0.591495,0.133123,0.207859,0.582926,0.091822,0.148584,0.599176,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,NadigOConner2024_jurkat,ZBTB17,0.690301,0.902702,0.492744,0.004280,0.020213,0.538405,0.017389,0.071384,0.530836,0.017037,0.062964,0.530956,"Nonzero-Rate, Uniform",0.311162,0.505305,NaN
665,ReplogleWeissman2022_K562_essential,ZBTB17,0.000632,0.003477,0.574356,0.996170,0.998774,0.438496,0.940080,0.998717,0.464126,0.987185,0.999916,0.448527,"Correlation, Specificity",0.002679,0.012503,Correlation
666,ReplogleWeissman2022_K562_gwps,ZBTB17,0.079301,0.382584,0.522973,0.964612,0.999996,0.470556,0.945989,0.999032,0.473811,0.989105,0.999999,0.462619,NaN,NaN,NaN,NaN
667,ReplogleWeissman2022_rpe1,ZBTB17,0.006653,0.014804,0.543304,0.539128,0.639765,0.498282,0.120939,0.195701,0.520473,0.681783,0.749119,0.491731,"Correlation, Specificity",0.349247,0.618665,NaN


## 3. Priors Supplementary Files

Separate tables are generated for the AllAvailableTFs and CommonTFsOnly analyses. Each table includes a Best3 block from the No-Ensemble DeLong comparison and a Best4 block from the With-Ensemble DeLong comparison.

In [5]:
prior_names = {
    "causalpath": "CausalPath",
    "collectri": "CollecTRI",
    "dorothea": "DoRothEA",
    "ensemble": "Ensemble",
}

def build_priors_supplementary_table(
    mwu_path,
    best3_delong_path,
    best4_delong_path,
):
    metric_names = {
        "P_Value": "MWU-pvalue",
        "Adjusted_P_Value": "MWU-adjusted-pvalue",
        "ROC_AUC": "ROC-AUC",
    }

    mwu_df = pd.read_csv(mwu_path, sep="\t")
    mwu_metrics = mwu_df.reindex(
        columns=["Dataset", "TF", "Prior", *metric_names]
    )
    ordered_columns = [
        (metric, prior) for prior in prior_names for metric in metric_names
    ]
    prior_values = (
        mwu_metrics[mwu_metrics["Prior"].isin(prior_names)]
        .set_index(["Dataset", "TF", "Prior"])[list(metric_names)]
        .unstack("Prior")
        .reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    )
    prior_values.columns = [
        f"{prior_names[prior]}_{metric_names[metric]}"
        for metric, prior in prior_values.columns
    ]

    def build_delong_block(delong_path, block_name):
        delong_df = pd.read_csv(delong_path, sep="\t")
        summary = delong_df[
            [
                "Dataset",
                "TF",
                "Top_Prior",
                "Second_Prior",
                "Comparison_Type",
                "DeLong_P_Value",
                "DeLong_P_Value_FDR_BH",
                "DeLong_Significant_FDR_BH",
            ]
        ].copy()
        top_prior = summary["Top_Prior"].map(prior_names)
        second_prior = summary["Second_Prior"].map(prior_names)
        pair_column = f"{block_name}_Top, Second"
        summary[pair_column] = top_prior.where(
            second_prior.isna(), top_prior + ", " + second_prior
        )
        winner_column = f"{block_name}_Winner"
        summary[winner_column] = pd.NA
        single_option = summary["Comparison_Type"].eq("single_available_significant")
        summary.loc[single_option, winner_column] = top_prior[single_option]
        delong_winner = summary["DeLong_Significant_FDR_BH"].eq(True) & ~single_option
        summary.loc[delong_winner, winner_column] = top_prior[delong_winner]
        return summary.drop(
            columns=[
                "Top_Prior",
                "Second_Prior",
                "Comparison_Type",
                "DeLong_Significant_FDR_BH",
            ]
        ).rename(
            columns={
                "DeLong_P_Value": f"{block_name}_DeLongs-pvalue",
                "DeLong_P_Value_FDR_BH": f"{block_name}_DeLongs-adjusted-pvalue",
            }
        )

    best3_summary = build_delong_block(best3_delong_path, "Best3")
    best4_summary = build_delong_block(best4_delong_path, "Best4")
    result = (
        prior_values.reset_index()
        .merge(best3_summary, on=["Dataset", "TF"], how="left")
        .merge(best4_summary, on=["Dataset", "TF"], how="left")
    )

    return result[
        ["Dataset", "TF"]
        + list(prior_values.columns)
        + [
            "Best3_Top, Second",
            "Best3_DeLongs-pvalue",
            "Best3_DeLongs-adjusted-pvalue",
            "Best3_Winner",
            "Best4_Top, Second",
            "Best4_DeLongs-pvalue",
            "Best4_DeLongs-adjusted-pvalue",
            "Best4_Winner",
        ]
    ]

prior_supplementary_configs = {
    "AllAvailableTFs": {
        "mwu_filename": "MWU_merged_AllAvailableTFs.tsv",
        "best3_delong_filename": "DeLong_top2_No-Ensemble_AllAvailableTFs.tsv",
        "best4_delong_filename": "DeLong_top2_With-Ensemble_AllAvailableTFs.tsv",
    },
    "CommonTFsOnly": {
        "mwu_filename": "MWU_merged_CommonTFsOnly.tsv",
        "best3_delong_filename": "DeLong_top2_No-Ensemble_CommonTFsOnly.tsv",
        "best4_delong_filename": "DeLong_top2_With-Ensemble_CommonTFsOnly.tsv",
    },
}

prior_table_summary = []
for tf_set, filenames in prior_supplementary_configs.items():
    priors_delongs_df = build_priors_supplementary_table(
        mwu_path=f"results/Priors_MWU-Delongs/{filenames['mwu_filename']}",
        best3_delong_path=f"results/Priors_MWU-Delongs/{filenames['best3_delong_filename']}",
        best4_delong_path=f"results/Priors_MWU-Delongs/{filenames['best4_delong_filename']}",
    )
    priors_delongs_df = sort_supplementary_table(priors_delongs_df)
    output_path = f"{supplementary_dir}/Priors_Delongs_Comparison_Table_{tf_set}.tsv"
    priors_delongs_df.to_csv(output_path, sep="\t", index=False)
    prior_table_summary.append({"TF set": tf_set, "Rows": len(priors_delongs_df), "File": output_path})

pd.DataFrame(prior_table_summary)

,TF set,Rows,File
0,AllAvailableTFs,1300,results/Supplementary_Files/Priors_Delongs_Com...
1,CommonTFsOnly,225,results/Supplementary_Files/Priors_Delongs_Com...
